# Stage 1: Playlist Audit & Inventory
This notebook audits the target YouTube playlist to extract video metadata, identify accessible videos, detect private/deleted videos, and estimate resource/quota utilization before performing data collection.

In [ ]:
# Import custom settings and auditor
import sys
import os
from pathlib import Path

# Ensure path is configured correctly
import sys, os
from pathlib import Path
cwd = Path(os.getcwd()).resolve()
base_dir = cwd if (cwd / 'config').exists() else (cwd.parent if (cwd.parent / 'config').exists() else cwd)
if str(base_dir) not in sys.path: sys.path.insert(0, str(base_dir))

import yaml
from config import settings
from src.audit.playlist_auditor import audit_playlist, generate_audit_reports

## Load Target Playlist ID from Registry

In [ ]:
with open(settings.PLAYLIST_REGISTRY_PATH, "r") as f:
    registry = yaml.safe_load(f)

playlist = [p for p in registry["playlists"] if p["target"]][0]
playlist_id = playlist["id"]
print(f"Target Playlist: {playlist['name']}")
print(f"Playlist ID: {playlist_id}")

## Run Playlist Auditor
*(Make sure `YOUTUBE_API_KEY` is loaded in your `.env` file or environment variables)*

In [ ]:
try:
    videos = audit_playlist(playlist_id)
    print(f"Successfully audited. Found {len(videos)} videos.")
except ValueError as e:
    print(f"Error: {e}")
    print("Using mock data for pipeline verification.")
    # Generate dummy videos for offline testing
    videos = [
        {
            "video_id": f"video_mock_{i}",
            "title": f"Belajar Sains Kok Bisa Part {i}",
            "description": "Penjelasan sains menarik tentang fenomena alam.",
            "published_at": "2026-01-01T00:00:00Z",
            "privacy_status": "public",
            "position": i,
            "is_accessible": True
        } for i in range(1, 11)
    ]
    videos[4]["title"] = "Private video"  # Add a mock private video
    videos[4]["is_accessible"] = False
    videos[4]["privacy_status"] = "private"

## Generate Audit Reports
This creates `reports/01_playlist_report.md` and `reports/02_video_inventory.csv`.

In [ ]:
summary = generate_audit_reports(playlist_id, videos)
print("Reports Generated Successfully:")
print(f"- CSV inventory: {summary['csv_path']}")
print(f"- Markdown report: {summary['md_path']}")
print(f"- Public/Accessible Videos: {summary['accessible_videos']}/{summary['total_videos']}")